In [1]:
using Glob 
using DelimitedFiles, DataFrames, EzXML
using Dates
using LinearAlgebra
using ForwardDiff
using Statistics
using Interpolations
using Rotations
using Unitful
using ThreadsX
using WENO4
using Polyester
using CoordinateTransformations
import PhysicalConstants.CODATA2018: m_p, m_e, e, ε_0, μ_0, c_0, G, k_B
const M_Mars = 0.64169e24u"kg"
const R_Mars = 3389.5u"km"  #m
const au = 149597870.7u"km"

include("../../src/scripts/MAVEN_load.jl")
include("../../src/scripts/MAVEN_STATIC.jl")
include("../../src/scripts/SphericalSplines.jl")
using .MAVEN_STATIC
using .MAVEN_load
using .SphericalSplines

date = Date(2022, 8, 24)
datestr = Dates.format.(date, "yyyymmdd")
year_num = Dates.format(date, "yyyy")
month_num = Dates.format(date, "mm")

ts = DateTime(date) + Dates.Hour(7) + Dates.Minute(26)
te = ts + Dates.Minute(1)

data_dir = "E:\\Data"
output_dir = "E:\\Output"
MAVEN_path = joinpath(data_dir, "MAVEN")
MAG_path = joinpath(MAVEN_path, "MAG", "l2", year_num, month_num)
STATIC_path = joinpath(MAVEN_path, "STATIC", "l2", year_num, month_num)
MAVEN_output_path = joinpath(output_dir, "MAVEN")
STATIC_output_path = joinpath(MAVEN_output_path, "STATIC")
VDF_path = joinpath(STATIC_output_path, "VDF", year_num, month_num)
mkpath(VDF_path)
MAG_file = glob("mvn_mag_l2*$datestr*.sts", MAG_path)
STA_file = glob("mvn_sta_l2*d1*32e4d16a8m*$datestr*.cdf", STATIC_path)
println("MAG file: ", MAG_file)
println("STATIC file: ", STA_file)
println("Output path: ", VDF_path)

MVN_mag2c1hz = MAVEN_load.load_mag_l2(MAG_file[1])
MVN_static = MAVEN_load.load_cdf(STA_file[1])
println("Data loaded.")

# MVN MAG
datejul = datetime2julian(DateTime(date))
MVN_magut1hz = MVN_mag2c1hz[:JulUTtime].-datejul
MVN_BMSO1hz = MVN_mag2c1hz[:B]
ind = ThreadsX.findall(ts .<= MVN_mag2c1hz[:epoch] .<= te)
MVN_Bmeanmso = dropdims(mean(Matrix(MVN_BMSO1hz[ind, :]), dims=(1,)), dims=(1,))

# MVN STATIC
MVN_staticswp = MVN_static[:swp_ind] |> Array 
MVN_staticswp .+= 1
MVN_statictime = MVN_static[:epoch] |> Array
MVN_staticjulian=datetime2julian.(DateTime.(MVN_statictime))
MVN_staticeflux = MVN_static[:eflux] |> Array
MVN_staticmass = MVN_static[:mass_arr] |> Array
MVN_staticenergy = MVN_static[:energy] |> Array
MVN_statictheta = MVN_static[:theta] |> Array
MVN_staticphi = MVN_static[:phi] |> Array
MVN_staticdenergy = MVN_static[:denergy] |> Array
MVN_staticdtheta = MVN_static[:dtheta] |> Array
MVN_staticdphi = MVN_static[:dphi] |> Array
MVN_staticdomega = MVN_static[:domega] |> Array
MVN_staticvelocity = sqrt.(MVN_staticenergy.*(2.0/ustrip(u"eV", m_p*c_0^2))./MVN_staticmass).*ustrip(u"km/s", c_0)
MVN_staticpos_sc_mso = MVN_static[:pos_sc_mso] |> Array
MVN_staticpos_sc_mso /= ustrip(u"km", R_Mars)

MVN_staticquat_mso = MVN_static[:quat_mso] |> Array
MVN_staticquat_sc = MVN_static[:quat_sc] |> Array
MVN_staticmsotrans = [QuatRotation(quat) for quat in eachcol(MVN_staticquat_mso')]
MVN_staticind = ThreadsX.findall(ts .<= MVN_statictime .<= te)
MVN_staticswpind = MVN_staticswp[MVN_staticind[1]]
MVN_staticefluxmean = dropdims(mean(MVN_staticeflux[MVN_staticind, :, :, :], dims = (1, ) ), dims=(1, ))
MVN_staticpsdmean = ustrip(u"cm^-3 * km^-3 * s^3", 0.5 * 1u"cm^-2 * s^-1" * (m_p/1u"eV")^2)  .* MVN_staticefluxmean .* (MVN_staticmass[:,:,:,MVN_staticswpind,1]./ MVN_staticenergy[:,:,:,MVN_staticswpind,1]).^2

MVN_staticefluxmid = similar(MVN_staticefluxmean)
MVN_staticefluxuquant = similar(MVN_staticefluxmean)
MVN_staticefluxlquant = similar(MVN_staticefluxmean)
for i in axes(MVN_staticeflux, 2), j in axes(MVN_staticeflux, 3), k in axes(MVN_staticeflux, 4)
    data = MVN_staticeflux[MVN_staticind, i, j, k]
    MVN_staticefluxmid[i, j, k] = quantile(data, 0.5)
    MVN_staticefluxuquant[i, j, k] = quantile(data, 0.75)
    MVN_staticefluxlquant[i, j, k] = quantile(data, 0.25)
end

MVN_staticindmean = round(Int64, mean(MVN_staticind))
MVN_statictimemean = MVN_statictime[MVN_staticindmean]
MVN_staticjulianmean = MVN_staticjulian[MVN_staticindmean]
MVN_static2msomean = QuatRotation(MVN_staticquat_mso[MVN_staticindmean, :])
MVN_Bmeanstatic = MVN_static2msomean' * MVN_Bmeanmso
MVN_parastatic = MVN_Bmeanstatic ./ norm(MVN_Bmeanstatic)
swi, swj =2, 12
ind = argmax(MVN_staticefluxmean[1, (swj-1)*4+swi, :] ./ MVN_staticenergy[1, 1, :, MVN_staticswpind, 1].^2)
MVN_staticvswrough = CartesianFromSpherical()(CoordinateTransformations.Spherical(
    MVN_staticvelocity[1, (swj-1)*4+swi, ind, MVN_staticswpind, 1], 
    deg2rad(MVN_staticphi[1, (swj-1)*4+swi, ind, MVN_staticswpind, 1]), 
    deg2rad(MVN_statictheta[1, (swj-1)*4+swi, ind, MVN_staticswpind, 1])))
MVN_staticdirections = [CartesianFromSpherical()(CoordinateTransformations.Spherical(1.0, deg2rad(MVN_staticphi[1, ij, ind, MVN_staticswpind, 1]), deg2rad(MVN_statictheta[1, ij, ind, MVN_staticswpind, 1]))) for ij in 1:64]
MVN_staticss = []
Snan = SphericalSplines.InterpolationSpline(stack(MVN_staticdirections; dims=1), fill(-30.0, 64))
for k in 1:32
    ind = findall(MVN_staticefluxmean[1, :, k] .> 0.0)
    if !isempty(ind)
        S = SphericalSplines.InterpolationSpline(stack(MVN_staticdirections[ind]; dims=1), log10.(MVN_staticefluxmean[1, ind, k]))
        push!(MVN_staticss, S)
    else
        push!(MVN_staticss, Snan)
    end  
end
MVN_perpstatic1 = cross(MVN_parastatic, MVN_staticvswrough)
MVN_perpstatic1 = MVN_perpstatic1 ./ norm(MVN_perpstatic1)
MVN_perpstatic2 = cross(MVN_perpstatic1, MVN_parastatic)
MVN_static2mfa = transpose(hcat(MVN_perpstatic1, MVN_parastatic, MVN_perpstatic2))

# 预计算有效能道（排除Snan）
valid_channels = [k for k in 6:32 if MVN_staticss[k] !== Snan]
valid_logv = log10.(MVN_staticvelocity[1, 1, valid_channels, MVN_staticswpind, 1])
println("Valid WENO4 channels: $(length(valid_channels))/27")

# ═══════════════════════════════════════════════════════
#  插值函数
# ═══════════════════════════════════════════════════════
function MVN_STATIC_eflux_interp(v3)
    vs = SphericalFromCartesian()(v3) 
    vtot = vs.r
    lon = vs.θ
    lat = vs.ϕ
    v_lo = MVN_staticvelocity[1, 1, valid_channels[1], MVN_staticswpind, 1]
    v_hi = MVN_staticvelocity[1, 1, valid_channels[end], MVN_staticswpind, 1]
    if (abs(rad2deg(lat)) <= 45.0) & (v_lo ≥ vtot ≥ v_hi)
        dir = v3 ./ vtot
        nvalid = length(valid_channels)
        logeflux = Vector{Float64}(undef, nvalid)
        for (idx, k) in enumerate(valid_channels)
            S = MVN_staticss[k]
            logeflux[idx] = S(dir)
        end
        x = reverse(valid_logv)
        y = reverse(logeflux)
        return 10.0^interpolate_weno4([log10(vtot)], x, y)[1]
    else
        return NaN64
    end
end

# ═══════════════════════════════════════════════════════
#  任意斜面截面插值函数
#  plane_origin: 平面上的一点 (3D向量, MFA坐标)
#  plane_normal: 平面法向量 (3D向量, 会被归一化)
#  range_s, range_t: 截面两个方向的范围 (km/s)
#  n_s, n_t: 两个方向的采样点数
#  返回: (grid_s, grid_t, eflux_2d)
#    grid_s, grid_t: 2D网格坐标 (用于绘图)
#    eflux_2d: 2D截面数据 (n_s × n_t)
# ═══════════════════════════════════════════════════════
function slice_plane_interp(plane_origin, plane_normal;
        range_s = (-1000.0, 1000.0), range_t = (-1000.0, 1000.0),
        n_s = 200, n_t = 200)

    # 归一化法向量
    n = plane_normal ./ norm(plane_normal)

    # 构建平面内正交基 (Gram-Schmidt)
    # 选一个不平行于n的辅助向量
    if abs(n[1]) < 0.9
        aux = [1.0, 0.0, 0.0]
    else
        aux = [0.0, 1.0, 0.0]
    end
    u = cross(aux, n)
    u = u ./ norm(u)
    v = cross(n, u)  # v = n × u, 保证右手系

    # 2D网格
    s_vals = collect(range(range_s[1], range_s[2], length = n_s))
    t_vals = collect(range(range_t[1], range_t[2], length = n_t))

    # 插值
    eflux_2d = Array{Float64, 2}(undef, n_s, n_t)
    @batch for it in 1:n_t
        for is in 1:n_s
            v3 = plane_origin .+ s_vals[is] .* u .+ t_vals[it] .* v
            eflux_2d[is, it] = MVN_STATIC_eflux_interp(v3)
        end
    end

    return s_vals, t_vals, eflux_2d, u, v
end

# ═══════════════════════════════════════════════════════
#  3D直角网格插值 (与vdf_3D.jl相同)
# ═══════════════════════════════════════════════════════
vpara = collect(range(-1000.0, 1000.0, step = 10.0))
vperp1 = collect(range(-1000.0, 1000.0, step = 10.0))
vperp2 = collect(range(-1000.0, 1000.0, step = 10.0))
MVN_static_eflux_rect = Array{Float64, 3}(undef, length(vperp1), length(vpara), length(vperp2))
@batch for k in 1:length(vperp2)
    for j in 1:length(vpara)
        for i in 1:length(vperp1)
            v3 = Array(vpara[j] .* MVN_parastatic .+ vperp1[i] .* MVN_perpstatic1 .+ vperp2[k] .* MVN_perpstatic2)
            MVN_static_eflux_rect[i, j, k] = MVN_STATIC_eflux_interp(v3)
        end
    end
end

MAG file: ["E:\\Data\\MAVEN\\MAG\\l2\\2022\\08\\mvn_mag_l2_2022236ss_20220824_v01_r01.sts"]
STATIC file: ["E:\\Data\\MAVEN\\STATIC\\l2\\2022\\08\\mvn_sta_l2_d1-32e4d16a8m_20220824_v02_r03.cdf"]
Output path: E:\Output\MAVEN\STATIC\VDF\2022\08
Data loaded.
Valid WENO4 channels: 22/27


In [2]:
# ═══════════════════════════════════════════════════════
#  截面定义 (修改这里的参数来改变截面)
# ═══════════════════════════════════════════════════════
# 截面过原点, 法向量为 (1, 1, 1) 方向 → 即 v_para = v_perp1 = v_perp2 的斜面
# 也可以设为其他平面, 例如:
#   平行于B的平面: plane_normal = MVN_parastatic
#   平行于E的平面: plane_normal = MVN_perpstatic1
#   任意角度: plane_normal = normalize([1.0, 1.0, 1.0])

plane_origin = [0.0, 0.0, 0.0]                # 平面过原点
plane_normal = normalize([3.0, 0, -1.0])     # 法向量 (1,1,1)方向

println("Computing oblique slice...")
println("  Plane normal (MFA): $(round.(plane_normal, digits=3))")
s_vals, t_vals, eflux_slice, u_vec, v_vec = slice_plane_interp(
    plane_origin, plane_normal;
    range_s = (-1000.0, 1000.0), range_t = (-1000.0, 1000.0),
    n_s = 200, n_t = 200)

println("  Slice shape: $(size(eflux_slice))")
finite_count = count(isfinite, eflux_slice)
println("  Finite values: $finite_count / $(length(eflux_slice))")

Computing oblique slice...
  Plane normal (MFA): [0.949, 0.0, -0.316]
  Slice shape: (200, 200)
  Finite values: 14224 / 40000


In [5]:
ts = DateTime(2022, 8, 24, 7, 27, 0)
# ═══════════════════════════════════════════════════════
#  激波法向计算 (MVA最小方差分析)
# ═══════════════════════════════════════════════════════
function MVA(b::AbstractMatrix)
    nb = size(b, 1)
    bm = mean(b, dims=1)
    muv = b' * b ./ nb - bm' * bm
    return eigen(muv)
end

shock_time = DateTime(2022, 8, 24, 7, 27, 0)
shock_window = Second(10)  # 激波前后各10秒
t_shock_start = shock_time - shock_window
t_shock_end = shock_time + shock_window

# 取激波穿越前后的MAG数据
ind_shock = ThreadsX.findall(t_shock_start .<= MVN_mag2c1hz[:epoch] .<= t_shock_end)
B_shock = MVN_mag2c1hz[:B][ind_shock, :]

# MVA求激波法向
eig_shock = MVA(B_shock)
# 最小特征值对应的特征向量 = 激波法向 (MSO坐标)
n_shock_mso = eig_shock.vectors[:, 1]  # 最小方差方向

# 确保法向大致指向太阳方向 (x>0)
if n_shock_mso[1] < 0
    n_shock_mso = -n_shock_mso
end

# 转换到MFA坐标系
n_shock_mfa = MVN_static2mfa * n_shock_mso

println("Shock normal (MSO): $(round.(n_shock_mso, digits=3))")
println("Shock normal (MFA): $(round.(n_shock_mfa, digits=3))")
println("Eigenvalues: $(round.(eig_shock.values, digits=2))")

Shock normal (MSO): Float32[0.803, -0.568, -0.181]
Shock normal (MFA): [0.075, -0.608, -0.79]
Eigenvalues: Float32[0.45, 4.52, 6.04]


In [13]:
# ═══════════════════════════════════════════════════════
#  绘图: 左=3D体渲染 + 截面, 右=2D截面展开图
# ═══════════════════════════════════════════════════════
using GLMakie
GLMakie.activate!()

size_inches = (20, 9)
size_pt = 72 .* size_inches
fig = Figure(size = size_pt, fontsize = 15, figure_padding = (5, 10, 9, 8))
title_str = "MAVEN STATIC H⁺ VDF " * Dates.format(ts, "yyyymmdd HH:MM") * "-" * Dates.format(te, "HH:MM")
fig_name = "MAVEN_STATIC_H+_VDF_slice_" * Dates.format(ts, "yyyymmddTHHMMSS") *
    "-" * Dates.format(te, "HHMMSS") * ".png"

# ── 左图: 3D场景 + 斜面截面 ──
Label(fig[1, 1, Top()], title_str * "\n3D + Oblique Slice", fontsize = 16, tellheight = true)
ls1 = LScene(fig[1, 1])

# 画几层 vpara 切片作为参考背景 (半透明)
for ipara in 1:2:length(vpara)
    heatmap!(ls1, vperp1, vperp2, MVN_static_eflux_rect[:, ipara, :],
        transformation = (:xz, vpara[ipara]),
        colormap = :gist_earth, colorrange = (1e4, 1e8), colorscale = log10)
end

# 在3D场景中画斜面截面
# 构建截面的4个角点 (在MFA坐标中)
# corner_s = [-1000.0, 1000.0, 1000.0, -1000.0]
# corner_t = [-1000.0, -1000.0, 1000.0, 1000.0]
# # 转为3D坐标 (用于mesh!的vertices)
# function plane_point(origin, u, v, s, t)
#     return origin .+ s .* u .+ t .* v
# end

# # 用mesh画截面平面 (半透明底色)
# # mesh! 需要Mesh primitive, 这里用 surface! 替代
# # 实际用 lines! 画截面边框
# border_s = vcat(range(-1000, 1000, length=50), fill(1000.0, 50), range(1000, -1000, length=50), fill(-1000.0, 50))
# border_t = fill(-1000.0, 50), vcat(fill(-1000.0, 50), range(-1000, 1000, length=50))  # 简化: 画边框线

# # 在3D场景中插入截面 (用 heatmap surface)
# # 把截面数据转为 surface! 可用的格式
# # surface! 需要 x, y, z 三个2D数组 + color 2D数组
# n_s, n_t = size(eflux_slice)
# xs = zeros(n_s, n_t)
# ys = zeros(n_s, n_t)
# zs = zeros(n_s, n_t)
# for it in 1:n_t, is in 1:n_s
#     pt = plane_origin .+ s_vals[is] .* u_vec .+ t_vals[it] .* v_vec
#     xs[is, it] = pt[1]  # perp1
#     ys[is, it] = pt[2]  # para
#     zs[is, it] = pt[3]  # perp2
# end

# # 截面边框线 (在3D场景中可视化截面位置)
# # 构建截面4条边的3D坐标
# function make_plane_corners(origin, u, v, s_range, t_range)
#     s_min, s_max = s_range
#     t_min, t_max = t_range
#     corners = [
#         origin .+ s_min .* u .+ t_min .* v,
#         origin .+ s_max .* u .+ t_min .* v,
#         origin .+ s_max .* u .+ t_max .* v,
#         origin .+ s_min .* u .+ t_max .* v,
#         origin .+ s_min .* u .+ t_min .* v,  # 闭合
#     ]
#     return [Point3f(c...) for c in corners]
# end

# plane_corners = make_plane_corners(plane_origin, u_vec, v_vec, (-1000, 1000), (-1000, 1000))
# lines!(ls1, plane_corners, color = :cyan, linewidth = 1.5, linestyle = :dash)

# 太阳风速度方向箭头 (MFA坐标)
MVN_vsw_mfa = MVN_static2mfa * MVN_staticvswrough  # 静态坐标→MFA坐标
ps_vsw = Point3f(0.0, 0.0, 0.0)
pe_vsw = Point3f(MVN_vsw_mfa...)
linesegments!(ls1, [ps_vsw, pe_vsw], color = :red, linewidth = 8, overdraw = true)
arrows3d!(ls1, ps_vsw, pe_vsw, argmode = :endpoint,
    color = :red, shaftradius = 0.1, tipradius = 0.2, tailradius = 0.06, overdraw = true)

# 反射方向计算: v_ref = v - 2(v·n)n
v_incident = MVN_vsw_mfa  # 入射方向 (指向激波面)
dot_vn = dot(v_incident, n_shock_mfa)
v_reflected = v_incident - 2.0 * dot_vn .* n_shock_mfa

# 反射方向箭头 (橙色)
ps_ref = Point3f(0.0, 0.0, 0.0)
pe_ref = Point3f(v_reflected...)
linesegments!(ls1, [ps_ref, pe_ref], color = :orange, linewidth = 6, overdraw = true)
arrows3d!(ls1, ps_ref, pe_ref, argmode = :endpoint,
    color = :orange, shaftradius = 0.08, tipradius = 0.15, tailradius = 0.05, overdraw = true)

# # 激波法向箭头 (白色, 更醒目)
# ps_shock = Point3f(0.0, 0.0, 0.0)
# pe_shock = Point3f(n_shock_mfa .* 600.0)  # 放大到600 km/s可见
# linesegments!(ls1, [ps_shock, pe_shock], color = :white, linewidth = 6, overdraw = true)
# arrows3d!(ls1, ps_shock, pe_shock, argmode = :endpoint,
#     color = :white, shaftradius = 0.08, tipradius = 0.15, tailradius = 0.05, overdraw = true)

# 激波面 (垂直于法向的平面, 紫色虚线边框)
# 构建平面内正交基
if abs(n_shock_mfa[1]) < 0.9
    aux_shock = [1.0, 0.0, 0.0]
else
    aux_shock = [0.0, 1.0, 0.0]
end
u_shock = cross(aux_shock, n_shock_mfa)
u_shock = u_shock ./ norm(u_shock)
v_shock = cross(n_shock_mfa, u_shock)

# 画激波面边框
shock_corners = [
    -1000.0 .* u_shock .- 1000.0 .* v_shock,
     1000.0 .* u_shock .- 1000.0 .* v_shock,
     1000.0 .* u_shock .+ 1000.0 .* v_shock,
    -1000.0 .* u_shock .+ 1000.0 .* v_shock,
    -1000.0 .* u_shock .- 1000.0 .* v_shock,  # 闭合
]
shock_corners_3f = [Point3f(c...) for c in shock_corners]
lines!(ls1, shock_corners_3f, color = :orange, linewidth = 2, linestyle = :dash, overdraw = true)

# ── 右图: 2D截面展开图 ──
Label(fig[1, 2, Top()], title_str * "\nSlice: n=$(round.(plane_normal, digits=2))", fontsize = 16, tellheight = true)
ax2 = Axis(fig[1, 2],
    xlabel = "s (km/s) ← $(round.(u_vec, digits=2))",
    ylabel = "t (km/s) ← $(round.(v_vec, digits=2))",
    aspect = 1)

hm2 = heatmap!(ax2, s_vals, t_vals, eflux_slice',
    colormap = :gist_earth, colorrange = (1e4, 1e8), colorscale = log10)

# 标注MFA轴方向
text!(ax2, "n̂ = $(round.(plane_normal, digits=2))", position = (-900, 900),
    fontsize = 12, color = :cyan, font = :bold)
text!(ax2, "û = $(round.(u_vec, digits=2))", position = (-900, 800),
    fontsize = 11, color = :yellow)
text!(ax2, "v̂ = $(round.(v_vec, digits=2))", position = (-900, 700),
    fontsize = 11, color = :magenta)
text!(ax2, "n̂_shock = $(round.(n_shock_mfa, digits=2))", position = (-900, 600),
    fontsize = 11, color = :orange, font = :bold)

Colorbar(fig[1, 3], hm2, label = "Differential Energy Flux")

save(joinpath(VDF_path, fig_name), fig)
println("Saved: ", joinpath(VDF_path, fig_name))

display(fig)
while isopen(fig.scene)
    sleep(0.1)
end

Saved: E:\Output\MAVEN\STATIC\VDF\2022\08\MAVEN_STATIC_H+_VDF_slice_20220824T072700-072700.png


In [ ]:
using Glob
using CommonDataFormat
using Dates
using Statistics, LinearAlgebra
using Rotations
using CairoMakie
using Interpolations
using Rotations
using Unitful
using ThreadsX
using WENO4
using Polyester
using CoordinateTransformations
import PhysicalConstants.CODATA2018: m_p, m_e, e, ε_0, μ_0, c_0, G, k_B
const M_Mars = 0.64169e24u"kg"
const R_Mars = 3389.5u"km"  #m
const au = 149597870.7u"km"
include("../../src/scripts/MAVEN_STATIC.jl")
include("../../src/scripts/MAVEN_load.jl")
include("../../src/scripts/MAVEN_plot.jl")
include("../../src/scripts/SphericalSplines.jl")
using .MAVEN_STATIC
using .MAVEN_load
using .MAVEN_plot
using .SphericalSplines

In [ ]:
date = Date(2022, 8, 24)
datestr = Dates.format.(date, "yyyymmdd")
year_num = Dates.format(date, "yyyy")
month_num = Dates.format(date, "mm")

data_dir = "E:\\Data"
output_dir = "E:\\Output"
MAVEN_path = joinpath(data_dir, "MAVEN")
MAG_path = joinpath(MAVEN_path, "MAG", "l2", year_num, month_num)
STATIC_path = joinpath(MAVEN_path, "STATIC", "l2", year_num, month_num)
MAVEN_output_path = joinpath(output_dir, "MAVEN")
STATIC_output_path = joinpath(MAVEN_output_path, "STATIC")
VDF_path = joinpath(STATIC_output_path, "VDF", year_num, month_num)
mkpath(VDF_path)
MAG_file = glob("mvn_mag_l2*$datestr*.sts", MAG_path)
STA_file = glob("mvn_sta_l2*d1*32e4d16a8m*$datestr*.cdf", STATIC_path)
println("MAG file: ", MAG_file)
println("STATIC file: ", STA_file)
println("Output path: ", VDF_path)

mag_data = MAVEN_load.load_mag_l2(MAG_file[1])
sta_data = MAVEN_load.load_cdf(STA_file[1])
println("Data loaded.")

In [ ]:
VDFS_data = Dict()
df_unit = ["(cm",superscript("-2")," s",superscript("-1")," sr",superscript("-1"),")"]

spice = Dict(
    :mass_range => [0, 2],
    :m_int => 1,
)
ion_vel = 160.0
spice_name = "H+"
colorrange = (1e-13, 1e-10)
# ------------------STA part-------------------------
V_30ev = MAVEN_STATIC.ion_energy2v(30, spice[:m_int]) / 1000
V_494ev = MAVEN_STATIC.ion_energy2v(494, spice[:m_int]) / 1000

ts = DateTime.(date + Dates.Hour(7) + Dates.Minute(26))
# time_stemp = ts .+ Dates.Minute(3) .* range(0, 3)
time_stemp = ts .+ Dates.Second(30) .* range(0, 3)

time_b = mag_data[:epoch]
B_ss = mag_data[:B]
position_mso = mag_data[:position]

fig_name = "MAVEN_STATIC_H+_VDF_" * Dates.format(time_stemp[1], "yyyymmddTHHMM")
df_unit = ["(cm",superscript("-2")," s",superscript("-1")," sr",superscript("-1"),")"]
spice = Dict(
    :mass_range => [0, 2],
    :m_int => 1,
)
ion_vel = 160.0
spice_name = "H+"
fig_VDF = Figure(resolution = (1000, 500))
ticks = [-600, 0, 600]
plot_range = (-800, 800)
colorbar_label = rich("STATIC ", spice_name, " VDF ", df_unit...)

In [ ]:
time_ind = findall((d -> time_stemp[1]<=d<=time_stemp[end]), sta_data[:epoch])
sta_time = sta_data[:epoch][time_ind]
sta_slip_data = [MAVEN_STATIC.static_slip(sta_data, i) for i in time_ind]
for (i, slip) in enumerate(sta_slip_data)
    MAVEN_STATIC.STA_count2df(slip)
    # MAVEN_STATIC.static_rotation(slip)
end
rotation_Q = [QuatRotation(slip[:quat_mso]) for slip in sta_slip_data]
time_b_ind = last.([findmin(d -> abs.(d .- statime), time_b) for statime in sta_time])
vsc = (position_mso[time_b_ind.+1, :] .- position_mso[time_b_ind.-1, :]) ./ 
    (datetime2unix.(time_b[time_b_ind.+1]) .- datetime2unix.(time_b[time_b_ind.-1]))
b0 = B_ss[time_b_ind, :]
vel, _, den = MAVEN_STATIC.sta_v_4d.(sta_slip_data; energy_range=[30, 1e8], spice...)
vel

In [ ]:
mag_data[:B][time_b_ind, :]

In [ ]:
II = 1
_, time_ind = findmin(d -> abs(d - time_stemp[II]), sta_data[:epoch])
sta_time = sta_data[:epoch][time_ind]
sta_slip_data = MAVEN_STATIC.static_slip(sta_data, time_ind)
sta_slip_data = MAVEN_STATIC.STA_count2df(sta_slip_data)
rotation_Q = QuatRotation(sta_slip_data[:quat_mso])
_, time_b_ind = findmin(d -> abs(d - sta_time), time_b)
vsc = (position_mso[time_b_ind.+1, :] .- position_mso[time_b_ind.-1, :]) ./ 
    (datetime2unix(time_b[time_b_ind.+1]) .- datetime2unix(time_b[time_b_ind.-1]))
b0 = B_ss[time_b_ind, :]
# b0 = MAVEN_STATIC.rotate_vector_with_quat_reverse(b0,rotation_Q)
b0 = rotation_Q' * b0
vel, _, den = MAVEN_STATIC.sta_v_4d(sta_slip_data; energy_range=[30, 1e8], spice...)
vel = vel .+ vsc
V_data = MAVEN_STATIC.static_slip_2_V(sta_slip_data; spice...)
v = V_data[:v]
v0 = sqrt.(sum(v[:, :, :] .^ 2, dims=3))[:, :, 1]
V_data[:v0] = v0
V_data[:x0] = rotation_Q' * sta_slip_data[:pos_sc_mso]
x0 = V_data[:x0]

v0 = reshape(V_data[:v0], 32 * 64)
v = reshape(V_data[:v], 32 * 64, 3)
dF = reshape(V_data[:df], 32 * 64) .* 1e-3 # IS化 # unit sec^3/(cm^-3 km^-3) to sec^3 m^-6
bins_sc = reshape(sta_slip_data[:bins_sc],64,1) .* ones(64,32)
mask_bins_sc = reshape(bins_sc, 32 * 64)

keys_range = [
        :energy,:denergy,:theta,:dtheta,:phi,:dphi,
    ]
range_data =Dict(
    :shape => size(V_data[:df]),
    :mode => "sta",
    :m_int => spice[:m_int],
)

In [ ]:
v

In [ ]:
using CairoMakie
fig_VDF = Figure(resolution = (500, 500))
ticks = [-200, 0, 200]
plot_range = (-300, 300)
title = Dates.format(sta_time, "HH:MM:SS")
# ax1 = Axis(fig_VDF[1+JJ2, II], aspect=2, limits=((plot_range[1], plot_range[2]), (0, plot_range[2])), xlabel=X_label, ylabel=Y_label, yticks=ticks, xticks=ticks, alignmode=Inside())
# hidexdecorations!(ax1, grid=false, ticks=false, label=false)
ax2 = Axis(fig_VDF[1, 1], title=title, aspect=1, yticks=ticks, xticks=ticks, alignmode=Inside())
hidexdecorations!(ax2, grid=false, ticks=false)
ax3 = Axis(fig_VDF[2, 1], aspect=1, yticks=ticks, xticks=ticks, alignmode=Inside())
linkxaxes!(ax2, ax3)
ax2, rot2, ind_mask = MAVEN_plot.VDF_2d_slip(
        ax2,v,dF; 
        normal_vectors=[rotation_Q'*v for v in [[1,0,0], [0,1,0]]], 
        colorrange=colorrange, vbluk=vel, plot_range=plot_range, vsc=vsc,magf=b0,return_rot_matrix=true, angle_range=[-30, 30])
    for i in keys_range
        local val = reshape(sta_slip_data[i][1,:,:],32*64)
        range_data[i] = val[ind_mask.*mask_bins_sc .==1]
    end
    ax2 = MAVEN_plot.VDF_2d_mask(ax2,range_data;rot = rot2, vsc=vsc)
    ax3, rot3, ind_mask = MAVEN_plot.VDF_2d_slip(
        ax3,v,dF; 
        normal_vectors=[rotation_Q'*v for v in [[1,0,0], [0,1,0]]], 
        colorrange=colorrange, vbluk=vel, plot_range=plot_range, vsc=vsc,magf=b0, return_rot_matrix=true, angle_range=[-30, 30])
    for i in keys_range
        local val = reshape(sta_slip_data[i][1,:,:],32*64)
        range_data[i] = val[ind_mask.*mask_bins_sc .==1]
    end
    ax3 = MAVEN_plot.VDF_2d_mask(ax3,range_data;rot = rot3, vsc=vsc)

    for i in 1:2
        rott = [rot2, rot3][i]
        ax = [ax2, ax3][i]
        xx = -x0
        xx = rott * xx * 100
        lines!(ax, [0, xx[1]], [0, xx[2]], color=:red, linewidth=3, linestyle=:dash) # 火星方向
        vel_local = rott * vel
        # scatter!(ax, vel_local[1], vel_local[2], color=:white, markersize=15,marker='X') # 速度矢量
        # bb = rott * b0 * 100
        # lines!(ax, [0, bb[1]], [0, bb[2]], color=:white, linewidth=3, linestyle=:dash)
        lines!(ax, Circle(Point2f(0, 0), V_30ev), color=:white) # 30eV的能量圈
        lines!(ax, Circle(Point2f(0, 0), V_494ev), color=:white) 
        # arrows!(ax, [Point2f(0, 0)],[Vec2f(vel_local[1],vel_local[2])]; color=:blue,arrowsize=15,arrowtail=15,linewidth=3,align = :tip) # 速度矢量arrowtail
        arrows!(ax, [Point2f(0, 0)],[Vec2f(vel_local[1],vel_local[2])]; color=:blue,arrowsize=15,linewidth=3)
    end
    # if II == 1
        # ax1.ylabel = Y_label
        # ax2.ylabel = Y_vdf_label
        # ax3.ylabel = Z_vdf_label
    # else
    #     # hideydecorations!(ax1, grid=false, ticks=false)
    #     hideydecorations!(ax2, grid=false, ticks=false)
    #     hideydecorations!(ax3, grid=false, ticks=false)
    # end
    # hidexdecorations!(ax3, grid=false, ticks=false)

Colorbar(fig_VDF[1:2, 5], limits=colorrange, colormap=:viridis, scale=log10)

fig_VDF

In [ ]:
time_b = mag_data[:epoch]
function nearest_time(t, sorted_times)
    i = searchsortedfirst(sorted_times, t)
    if i == 1
        return 1
    elseif i > length(sorted_times)
        return length(sorted_times)
    else
        # 比较左右两个邻居谁更近
        dt_lo = t - sorted_times[i-1]
        dt_hi = sorted_times[i] - t
        return dt_hi < dt_lo ? i : i-1
    end
end
time_b_ind = [nearest_time(t, time_b) for t in sta_time]
vsc = (position_mso[time_b_ind.+1, :] .- position_mso[time_b_ind.-1, :]) ./ 
    (datetime2unix(time_b[time_b_ind.+1]) .- datetime2unix(time_b[time_b_ind.-1]))
